# residual-skip-add — ex1: build a toy ResidualBlock with conditional shortcut

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `residual-skip-add`. Running the final beacon cell reports progress against the `CNN: Residual skip-connection add` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Residual skip-connection add` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`residual-skip-add`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "residual-skip-add"
DD_SUBTOPIC = "CNN: Residual skip-connection add"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Residual skip-connection add — quick refresher

A `ResidualBlock` adds its input back to its output:

```
out = self.conv_branch(x) + self.skip(x)        # the residual add
```

**Two cases for the skip branch:**

1. **Same shape** — when `in_channels == out_channels` AND `stride == 1`, the conv branch preserves shape. The skip is just the input:
   ```
   self.skip = nn.Identity()        # or just write x + self.conv(x)
   ```

2. **Different shape** — when channels change or `stride > 1`, the conv branch reshapes `(B, IC, H, W) → (B, OC, H', W')`. The skip branch needs a 1×1 projection to match:
   ```
   self.skip = nn.Conv2d(in_channels, out_channels,
                         kernel_size=1, stride=first_stride)
   ```
   This is the 1×1 conv whose ONLY job is shape adjustment (see the `1x1-conv-channel-reshape` atom for the per-pixel-Linear interpretation).

**Why the addition matters (vs concatenation).** Add preserves the channel count and keeps the gradient path FLAT: `∂out/∂x = ∂conv/∂x + 1`. That `+ 1` is the 'gradient highway' that lets ResNet train at hundreds of layers — gradients never vanish, they always have a direct route to every earlier layer.

**Why ReLU comes AFTER the add (not before).** The full block is `ReLU(BN(conv(x)) + skip(x))`. Putting ReLU after the add keeps the skip clean (no nonlinearity along the highway) and lets the conv branch learn to OFFSET its skip contribution if it wants the output to be negative.

### Exercise 1 — build a toy ResidualBlock with conditional shortcut

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the residual-block construction pattern: conv branch + skip branch summed; skip is `nn.Identity` when shapes match, else a `1×1 Conv2d` projection.
> Keywords: residual, skip, identity, 1x1-shortcut, resnet
> ```

**KCs targeted:** `residual-skip-add-pattern`, `conditional-1x1-shortcut`

Build a toy `ResidualBlock` module. The block has TWO parallel branches that get ADDED to form the output:

- **conv branch**: `nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=first_stride, padding=1)` — the shape-changer.
- **skip branch**: depends on whether shapes match:
  - If `in_channels == out_channels` AND `first_stride == 1` → `nn.Identity()`.
  - Otherwise → `nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=first_stride, padding=0)`.

Forward: `out = self.conv(x) + self.skip(x)`. No ReLU, no BN in this drill — we're isolating the skip-add pattern.

Skeleton:

```
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, first_stride=1):
        super().__init__()
        # 1. self.conv = nn.Conv2d(...)
        # 2. self.skip = nn.Identity() or nn.Conv2d(1x1, stride=first_stride)
        ...
    def forward(self, x):
        return self.conv(x) + self.skip(x)
```

**What the test checks.**
- Identity-shaped block (in=out, stride=1) has `self.skip` of type `nn.Identity`.
- Shape-changing block (in≠out OR stride>1) has `self.skip` of type `nn.Conv2d` with `kernel_size=(1,1)`.
- Forward pass produces the expected `(B, out_channels, H_out, W_out)` shape.
- Forward output equals `conv(x) + skip(x)` element-wise.
- Gradient flows back through BOTH branches (the gradient highway property).

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, first_stride: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=3, stride=first_stride, padding=1,
        )
        if in_channels == out_channels and first_stride == 1:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Conv2d(
                in_channels, out_channels,
                kernel_size=1, stride=first_stride, padding=0,
            )

    def forward(self, x: Tensor) -> Tensor:
        return self.conv(x) + self.skip(x)


<details><summary>Solution</summary>

```python
class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, first_stride: int = 1):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=3, stride=first_stride, padding=1,
        )
        if in_channels == out_channels and first_stride == 1:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Conv2d(
                in_channels, out_channels,
                kernel_size=1, stride=first_stride, padding=0,
            )

    def forward(self, x: Tensor) -> Tensor:
        return self.conv(x) + self.skip(x)
```

**Why `nn.Identity` and not just write `x + self.conv(x)`.** Using `nn.Identity` makes the skip branch a NAMED submodule that shows up in `repr(block)`, `state_dict`, and `named_modules()`. That uniform structure is what lets ResNet's constructor wire BlockGroups without branching on the in==out case — every block has a `.skip` attribute, just different types.

**Why the 1×1 conv on the skip when shapes differ.** Two tensors of different shape can't be added — you need a projection. The 1×1 conv is the cheapest possible projection: O(IC*OC) params per pixel, no spatial mixing. See the `1x1-conv-channel-reshape` atom for the per-pixel-Linear interpretation.

**Why the gradient highway works.** `∂(conv(x) + skip(x))/∂x = ∂conv/∂x + ∂skip/∂x`. For `skip = Identity`, that second term is just `1`. So even if `∂conv/∂x` is zero or tiny (dead channels, saturated ReLUs deeper in the conv branch), the gradient w.r.t. x has a guaranteed +1 floor — vanishing gradients become impossible. This is the single architectural reason 50+ layer CNNs trained at all in 2015.

**In real ResNet** the conv branch is `BN(Conv) → ReLU → BN(Conv)` and there's a ReLU after the add. Adding those layers is `module-composition` work — not the SKIP-ADD skill this drill exercises.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()